In [ ]:
from pathlib import Path

7 Step Plan:

1. Cache to disk
2. Load the cache back into RAM
3. Derive edge_index/node partitioning once per trajectory
4. Build the (trajectory_id, t) pair index
5. Manually build one HeteroData by hand, for a single example, to see it work
6. Wrap 4–5 into Dataset.__len__/__getitem__
7. Wire up DataLoader, pull one batch, confirm it's correctly batched

In [ ]:
from mgn.data_processing import (
    load_dataset,
    cache_raw_trajectories_to_disk,
    update_flag_simple_node_type_to_static,
)

# 1. Cache to disk
dataset_dir = Path("/home/jxn/dev/meshgraphnets/mgn/data/flag_simple")
splits = ["train", "valid", "test"]

for split in splits:
    ds = load_dataset(path=dataset_dir, split=split)

    dir = dataset_dir / "pytorch" / split

    cache_raw_trajectories_to_disk(dataset=ds, out_dir=dir)
    update_flag_simple_node_type_to_static(dir=dir)

In [ ]:
# 2. Load the cache back into RAM

import torch

train_dir = Path("/home/jxn/dev/meshgraphnets/mgn/data/flag_simple/pytorch/train")

loaded_pts = []

loaded_pts = [torch.load(pt_file) for pt_file in train_dir.rglob("*.pt")]

In [ ]:
print(loaded_pts[0]["cells"].shape)
print(loaded_pts[0]["mesh_pos"].shape)
print(loaded_pts[0]["world_pos"].shape)
print(loaded_pts[0]["node_type"].shape)

In [ ]:
from tqdm.auto import tqdm

# 3. Derive edge_index / node partitioning once per trajectory.

# Extract Edge Indices from raw triangles
for loaded_pt in tqdm(loaded_pts):
    # (1, 3028, 3)
    cells: torch.Tensor = loaded_pt["cells"]
    num_cells = cells.shape[1]

    # (1, 1579, 2)
    mesh_pos: torch.Tensor = loaded_pt["mesh_pos"]

    num_vertices = mesh_pos.shape[1]

    # edges are formed with vertices 0 -> 1 -> 2 -> 0

    # (2, num_subset_of_edges_with_duplicates)
    zero_to_one_edge_index = torch.cat((cells[:, :, 0], cells[:, :, 1]), dim=0)
    one_to_two_edge_index = torch.cat((cells[:, :, 1], cells[:, :, 2]), dim=0)
    two_to_zero_edge_index = torch.cat((cells[:, :, 2], cells[:, :, 0]), dim=0)

    # There will be duplicate edges in interior vertices, remove them.

    # (2, num_edges_with_duplicates)
    edge_index = torch.cat(
        (zero_to_one_edge_index, one_to_two_edge_index, two_to_zero_edge_index), dim=1
    )

    # 1. Put the sender and the receiver relationships in numerical order. smaller int -> greater int
    # (2, num_possibly_duplicated_edges)
    edge_index, _ = torch.sort(edge_index, dim=0)

    # 2. Remove duplicate edges
    # (2, num_unique_edges)
    edge_index = torch.unique(edge_index, dim=1)

    # Eulers Formula for planar graphs:
    # V - E + F_all = 2, where V is number of vertices, E is number of edges, F_all is
    # the number of all faces, including the infinite face outside the mesh.
    # V - E + F_cells + 1 = 2, as there is only one additional face, the infinite
    # thus, exterior face. V - E + F_cells = 1 => E = F + V - 1
    if not edge_index.shape[1] == num_cells + num_vertices - 1:
        raise ValueError("TODO, useful error")

    bidirectional_edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

    loaded_pt["edge_index"] = bidirectional_edge_index